In [ ]:
# === Setup ===
# Runtime: <1 minute fast, <2 minutes full on a typical CPU (estimate).
# Hardware: CPU ok; no GPU required.
# Network: none; all datasets are generated locally.
# Competition-safe: general profile; check the actual contest package/data policy.
# Cẩm nang P08: NumPy, pandas, sklearn, Matplotlib, joblib; no package installation.
import os
import random
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
random.seed(42)
np.random.seed(42)
FAST = os.environ.get('OAI_FAST_MODE', '0') == '1'
rng = np.random.default_rng(42)
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)


# EDA — Khám phá dữ liệu trước baseline

Starter: baseline chạy hết; hoàn thành TODO trước khi đọc solution.

## Data

Nhiều lần khám trên một người, một cột xuất hiện sau sự kiện, missing và ba bản ghi bị nhân đôi. Tạo split theo người trước phân tích target phục vụ model selection.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def make_patients(count, prefix):
    """Return a repeated-measurement frame (3*count, 6); target is synthetic."""
    patient = np.repeat(np.arange(count), 3)
    signal = rng.normal(size=count)
    x1 = signal[patient] + rng.normal(0, 0.35, len(patient))
    x2 = rng.normal(size=len(patient))
    y = (signal[patient] > 0.4).astype(int)
    frame = pd.DataFrame({'id': [f'{prefix}_{i}' for i in range(len(patient))],
                          'patient_id': [f'{prefix}_person_{i}' for i in patient],
                          'x1': x1, 'x2': x2, 'target': y, 'after_event': y})
    frame.loc[frame.index % 10 == 0, 'x2'] = np.nan
    return frame

raw = make_patients(120 if FAST else 240, 'train')
raw = pd.concat([raw, raw.iloc[:3]], ignore_index=True)
test = make_patients(24, 'test').drop(columns=['target', 'after_event'])
test_ids = test['id'].to_numpy()
# WHY: these are exact duplicated records, not all repeat visits of one person.
data = raw.drop_duplicates().reset_index(drop=True)
features = ['x1', 'x2']
tr, va = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
              .split(data, groups=data['patient_id']))
assert set(data.iloc[tr]['patient_id']).isdisjoint(data.iloc[va]['patient_id'])
assert set(data['patient_id']).isdisjoint(test['patient_id'])
assert raw.duplicated().sum() == 3


## EDA

Dự đoán: số dòng trùng là 3; missing x2 khoảng 10%. Phân tích target trên train partition. Public test chỉ kiểm schema/input.

In [ ]:
profile = pd.DataFrame({'dtype': raw.dtypes.astype(str),
                        'missing_rate': raw.isna().mean(),
                        'unique_values': raw.nunique(dropna=False)})
train_part = data.iloc[tr]
class_share = train_part['target'].value_counts(normalize=True).sort_index()
print(profile.to_string())
print('exact duplicates:', int(raw.duplicated().sum()))
print('training class shares:', class_share.to_dict())
assert not test[features].isna().all().any()
# TODO: document the collection time of every candidate feature.
# TODO: add a train/test distribution comparison without test labels.


## Preprocess → Model

Median imputation và scaling nằm trong pipeline để mỗi fit chỉ học statistics từ train. Logistic regression là baseline nhỏ.

In [ ]:
def build_model():
    """Return an unfitted pipeline accepting a numeric frame (n, d)."""
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                         LogisticRegression(max_iter=500, random_state=42))

model = build_model()


## Train

In [ ]:
model.fit(data.iloc[tr][features], data.iloc[tr]['target'])


## Evaluate

Metric của lab là Macro F1 trên group hold-out.

In [ ]:
valid_pred = model.predict(data.iloc[va][features])
baseline_score = f1_score(data.iloc[va]['target'], valid_pred, labels=[0, 1],
                          average='macro', zero_division=0)
comparison = [{'feature_set': 'available_at_prediction', 'macro_f1': baseline_score}]
assert 0 <= baseline_score <= 1


In [ ]:
print(pd.DataFrame(comparison).to_string(index=False))
profile.to_json(OUT / 'eda_profile_starter.json', orient='table', indent=2)


## Submit

Sau khi khóa feature set, refit pipeline sạch trên toàn labeled data. Test chỉ dùng predict.

In [ ]:
model = build_model().fit(data[features], data['target'])
test_pred = model.predict(test[features])


In [ ]:
# WHY: validate the file read from disk, not only the in-memory frame.
submission = pd.DataFrame({'id': test_ids, 'label': test_pred})
assert submission.columns.tolist() == ['id', 'label']
assert len(submission) == len(test_ids)
assert submission['id'].is_unique
assert submission['label'].isin([0, 1]).all()
submission.to_csv(OUT / 'submission_starter.csv', index=False)
reloaded = pd.read_csv(OUT / 'submission_starter.csv')
assert reloaded['id'].tolist() == list(test_ids)
assert reloaded['label'].tolist() == list(test_pred)
print('submission rows:', len(reloaded))


## Observation → Why → Postmortem

Ghi ba findings, evidence, quyết định xử lý và một giả thuyết chưa được kiểm chứng. Làm E-1/T-1/O-1 trong exercises; score synthetic không phải benchmark đề thật.